In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import re
import pandas as pd


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

print("Loading ProtGPT2...")
tokenizer = GPT2Tokenizer.from_pretrained("nferruz/ProtGPT2")

model = GPT2LMHeadModel.from_pretrained(
    "nferruz/ProtGPT2",
    torch_dtype=torch.float16
).to(device)

model.eval()

Using device: cuda
Loading ProtGPT2...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [ ]:
def is_valid_amp(seq):
    length = len(seq)
    positive_charge = seq.count("K") + seq.count("R")

    return (
        10 <= length <= 50
        and positive_charge >= 2
    )


In [ ]:
candidates = set()
target = 50

batch_size = 32  # Generate 32 sequences at once

while len(candidates) < target:

    input_ids = tokenizer.encode(
        "<|endoftext|>",
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            input_ids,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            repetition_penalty=1.2,
            num_return_sequences=batch_size,
            pad_token_id=0,
            eos_token_id=0
        )

    for output in outputs:

        decoded = tokenizer.decode(
            output,
            skip_special_tokens=True
        )

        seq = re.sub(
            r'[^ACDEFGHIKLMNPQRSTVWY]',
            '',
            decoded.upper()
        )

        if is_valid_amp(seq):
            candidates.add(seq)

    print(f"Collected: {len(candidates)}/{target}")

Collected: 1/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 2/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 3/50
Collected: 4/50
Collected: 4/50
Collected: 4/50
Collected: 5/50
Collected: 5/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 6/50
Collected: 7/50
Collected: 7/50
Collected: 7/50
Collected: 7/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collected: 8/50
Collecte

In [ ]:
candidates = list(candidates)[:target]

df = pd.DataFrame({
    "Sequence": candidates,
    "Length": [len(s) for s in candidates],
    "PositiveCharge": [
        s.count("K") + s.count("R")
        for s in candidates
    ]
})

df.to_csv("generated_amp_candidates.csv", index=False)

print("\nDone!")
print(df.head())


Done!
                                            Sequence  Length  PositiveCharge
0          MDTTIYALILLITFIIIIFVIVFIVLKLNKLNSNISNILKK      41               4
1                       MDNQNDNTHLDIAKKIIEESIQDKQENK      28               4
2       MADKNTYIDYNKYVQDLLDQFERQNYISKKEYEQIKEEYSIKYK      44               8
3  MSNIILSKDQLLDLCNRLIKESEQNQYSLNDLENIINELNKIKENI...      50               7
4    MPRTNRKDRKKRRKELKSKKSARKEEKRREKQSQKHHKTHKHPKNDH      47              22


NameError: name 'df' is not defined

In [ ]:
fasta_output = ""
for i, row in df.iterrows():
    fasta_output += f">AMP_candidate_{i+1}\n{row['Sequence']}\n"

print(fasta_output)

# Save it too
with open("candidates.fasta", "w") as f:
    f.write(fasta_output)

from google.colab import files
files.download("candidates.fasta")

>AMP_candidate_1
MDTTIYALILLITFIIIIFVIVFIVLKLNKLNSNISNILKK
>AMP_candidate_2
MDNQNDNTHLDIAKKIIEESIQDKQENK
>AMP_candidate_3
MADKNTYIDYNKYVQDLLDQFERQNYISKKEYEQIKEEYSIKYK
>AMP_candidate_4
MSNIILSKDQLLDLCNRLIKESEQNQYSLNDLENIINELNKIKENIKNKN
>AMP_candidate_5
MPRTNRKDRKKRRKELKSKKSARKEEKRREKQSQKHHKTHKHPKNDH
>AMP_candidate_6
MNDDIHRSICSDIYVSYFQLHNFFKYFFIKFCFICIFVIFK
>AMP_candidate_7
MLLGEKKIWESREFYFFYNTTMSNDKILKLAKENNITLSDIKKIIKEGNL
>AMP_candidate_8
MGTHIRHTFDPGTCPLCGGRDACAHPHTGWECQVCDEHFPSWPDDQE
>AMP_candidate_9
MLNFNDFIAILISAVLVGFVVYEVYQYIKKKKEKRKCGGCNCGLDSCDVN
>AMP_candidate_10
MTWLFFVLLACVPVAIFLVALAVRWSDRYDRERPAEPADPDGRR
>AMP_candidate_11
MTCSIGVARFPADGADADTLVKAADSALYRAKTEGRNRVCAAA
>AMP_candidate_12
MALSDAQIALISQYKSAGVSEQTIAKAFGISLRTAYKYLKLAKK
>AMP_candidate_13
MIHLIIIALVVVALVYYLYKKSNKGKCSSCPMNHDMCSDCSTHNKKN
>AMP_candidate_14
MAEKKFYKIQIIDGEEYKYGVCKTIREAIIYAKEKNPGYSVHVKEIEPI
>AMP_candidate_15
MGEFISNRKYYSICESDLKELIKYNLLIKEKKFYYYNDKLLFFYK
>AMP_candidate_16
MMNNKMMNMGFMMGSMVLGLGFLVLLIVVILLIVWA

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
v1_hits = [17, 21, 32, 42, 44, 50]

v2_hits = [1, 6, 8, 10, 13, 16, 17, 19, 20, 21, 22, 25, 29, 31, 34, 35, 39, 41, 42, 43, 44, 45, 49]

overlap = sorted(set(v1_hits) & set(v2_hits))
v2_only = sorted(set(v2_hits) - set(v1_hits))

print("Confirmed by both models:", overlap)
print("New hits from V2 only:", v2_only)

Confirmed by both models: [17, 21, 42, 44]
New hits from V2 only: [1, 6, 8, 10, 13, 16, 19, 20, 22, 25, 29, 31, 34, 35, 39, 41, 43, 45, 49]


In [ ]:
import pandas as pd
from google.colab import files

# Upload your CSV
uploaded = files.upload()

# Load it


Saving generated_amp_candidates (1).csv to generated_amp_candidates (1).csv


FileNotFoundError: [Errno 2] No such file or directory: 'generated_amp_candidates(1).csv'

In [ ]:
df = pd.read_csv('/content/generated_amp_candidates (1).csv')
print(f"Loaded {len(df)} sequences")
print(df.head())

Loaded 50 sequences
                                            Sequence  Length  PositiveCharge
0          MDTTIYALILLITFIIIIFVIVFIVLKLNKLNSNISNILKK      41               4
1                       MDNQNDNTHLDIAKKIIEESIQDKQENK      28               4
2       MADKNTYIDYNKYVQDLLDQFERQNYISKKEYEQIKEEYSIKYK      44               8
3  MSNIILSKDQLLDLCNRLIKESEQNQYSLNDLENIINELNKIKENI...      50               7
4    MPRTNRKDRKKRRKELKSKKSARKEEKRREKQSQKHHKTHKHPKNDH      47              22


In [ ]:
print(df.shape)
print(df.describe())
print(df.head())

(50, 3)
          Length  PositiveCharge
count  50.000000       50.000000
mean   43.720000        6.820000
std     5.852054        4.826331
min    28.000000        2.000000
25%    42.250000        4.000000
50%    44.000000        5.500000
75%    48.000000        7.750000
max    50.000000       26.000000
                                            Sequence  Length  PositiveCharge
0          MDTTIYALILLITFIIIIFVIVFIVLKLNKLNSNISNILKK      41               4
1                       MDNQNDNTHLDIAKKIIEESIQDKQENK      28               4
2       MADKNTYIDYNKYVQDLLDQFERQNYISKKEYEQIKEEYSIKYK      44               8
3  MSNIILSKDQLLDLCNRLIKESEQNQYSLNDLENIINELNKIKENI...      50               7
4    MPRTNRKDRKKRRKELKSKKSARKEEKRREKQSQKHHKTHKHPKNDH      47              22


In [ ]:
VALID = set('ACDEFGHIKLMNPQRSTVWY')

gold = [17, 21, 42, 44]
silver = [1, 6, 8, 10, 13, 16, 19, 20, 22, 25, 29, 31, 34, 35, 39, 41, 43, 45, 49]
all_hits = gold + silver

for i, row in df.iterrows():
    candidate_num = i + 1
    if candidate_num in all_hits:
        seq = row['Sequence'].upper().strip()
        cleaned = ''.join(c for c in seq if c in VALID)
        if len(cleaned) >= 10:
            print(cleaned)

MDTTIYALILLITFIIIIFVIVFIVLKLNKLNSNISNILKK
MNDDIHRSICSDIYVSYFQLHNFFKYFFIKFCFICIFVIFK
MGTHIRHTFDPGTCPLCGGRDACAHPHTGWECQVCDEHFPSWPDDQE
MTWLFFVLLACVPVAIFLVALAVRWSDRYDRERPAEPADPDGRR
MIHLIIIALVVVALVYYLYKKSNKGKCSSCPMNHDMCSDCSTHNKKN
MMNNKMMNMGFMMGSMVLGLGFLVLLIVVILLIVWAVNKLSGNK
MKAVLFFVLLFLALGSSFAEPCQKDQKCSQNTKCCTGVCKSWKCTSFC
MAGVVFLLVAVVLFAAGLGAIFVAWPLAAGIAVLLGAAALVLSARRGSRV
MDYNKQDYITFICIAFFMFGVFLFFRFIYKIYKSIKNKTTSNK
MSTKAKKKLIKKKKKLLKQKNLKPKLKKKPLNLQKKIKTKNTKKKKKK
MNWLITIAIIIAIGYGAYNFWSRGSKQGCCGGECSCSSEEKVEKK
MPFFFDQNKIQTHHLLHCYLCFRSICTLHQKYRFNNTYIYIYIYIF
MFNSADLLLAICFVTIIIVIAVVFLILYFVFRYVINKALRNNKR
MFDSILLSLVGLSFATTCVFVWVRTWGSRRYRWRKRMDRVAKEVDRIRKG
MGNNANTIWLIFILLLLLVIFILKKIISILFWVLIAVAIYYYFQE
MEIYVSVLLAILAFIAVFLIMRSVMKKSGGNCSCEGSCSECPCHKHEDK
MTIIFKCECDFEYKTTKKLAKCPHCGKSMNVKKIKKKEIKKENDEKKKKK
MTIIVWLLIFLVVMGVLGFIVQMFKSNK
MGWLYLVLGVVIVVGIILFVVAKTSGKPKTEKGNK
MLVLLVVVFLSFLLLIGIGLSIVTIVIGSLLIGGLLGFGAWFYYRRYQRD
MPTGQQFSIRYPRNNGPDRGCGWGRFCRCL
MGTFIHILIWAIIVLSIAALVVYFIYRMVKK
MEAPKHVRICARCNQPFEWRKK

In [ ]:
VALID = set('ACDEFGHIKLMNPQRSTVWY')

fasta_clean = ""
gold = [17, 21, 42, 44]
silver = [1, 6, 8, 10, 13, 16, 19, 20, 22, 25, 29, 31, 34, 35, 39, 41, 43, 45, 49]
all_hits = gold + silver

for i, row in df.iterrows():
    candidate_num = i + 1
    if candidate_num in all_hits:
        seq = row['Sequence'].upper().strip()
        # Remove any character that isn't a valid amino acid
        cleaned = ''.join(c for c in seq if c in VALID)
        if len(cleaned) >= 10:
            fasta_clean += f">AMP_candidate_{candidate_num}\n{cleaned}\n"

print(fasta_clean)


>AMP_candidate_1
MDTTIYALILLITFIIIIFVIVFIVLKLNKLNSNISNILKK
>AMP_candidate_6
MNDDIHRSICSDIYVSYFQLHNFFKYFFIKFCFICIFVIFK
>AMP_candidate_8
MGTHIRHTFDPGTCPLCGGRDACAHPHTGWECQVCDEHFPSWPDDQE
>AMP_candidate_10
MTWLFFVLLACVPVAIFLVALAVRWSDRYDRERPAEPADPDGRR
>AMP_candidate_13
MIHLIIIALVVVALVYYLYKKSNKGKCSSCPMNHDMCSDCSTHNKKN
>AMP_candidate_16
MMNNKMMNMGFMMGSMVLGLGFLVLLIVVILLIVWAVNKLSGNK
>AMP_candidate_17
MKAVLFFVLLFLALGSSFAEPCQKDQKCSQNTKCCTGVCKSWKCTSFC
>AMP_candidate_19
MAGVVFLLVAVVLFAAGLGAIFVAWPLAAGIAVLLGAAALVLSARRGSRV
>AMP_candidate_20
MDYNKQDYITFICIAFFMFGVFLFFRFIYKIYKSIKNKTTSNK
>AMP_candidate_21
MSTKAKKKLIKKKKKLLKQKNLKPKLKKKPLNLQKKIKTKNTKKKKKK
>AMP_candidate_22
MNWLITIAIIIAIGYGAYNFWSRGSKQGCCGGECSCSSEEKVEKK
>AMP_candidate_25
MPFFFDQNKIQTHHLLHCYLCFRSICTLHQKYRFNNTYIYIYIYIF
>AMP_candidate_29
MFNSADLLLAICFVTIIIVIAVVFLILYFVFRYVINKALRNNKR
>AMP_candidate_31
MFDSILLSLVGLSFATTCVFVWVRTWGSRRYRWRKRMDRVAKEVDRIRKG
>AMP_candidate_34
MGNNANTIWLIFILLLLLVIFILKKIISILFWVLIAVAIYYYFQE
>AMP_candidate_35
MEIYVSVLLAILAFIAV

1437